In [1]:
import spark

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WordCount") \
    .master('yarn') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 21:44:58 INFO SparkEnv: Registering MapOutputTracker
26/04/01 21:44:58 INFO SparkEnv: Registering BlockManagerMaster
26/04/01 21:44:58 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/01 21:44:58 INFO SparkEnv: Registering OutputCommitCoordinator


In [7]:
customer_data = [
"customer_id,name,city,state,country,registration_date,is_active",
"0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True",
"1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True",
"2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True",
"3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False",
"4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False",
"5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False",
]

In [8]:
data_rdd = spark.sparkContext.parallelize(customer_data)

In [9]:
data_rdd.getNumPartitions()

2

In [ ]:
# first() -returns the first element of RDD

In [11]:
header = data_rdd.first()
header

'customer_id,name,city,state,country,registration_date,is_active'

In [ ]:
# Filter () -> based on the condition its going to transform the dataset

In [12]:
data_rdd = data_rdd.filter(lambda row : row!=header)

In [13]:
data_rdd

PythonRDD[3] at RDD at PythonRDD.scala:53

In [ ]:
# Map() - It applies a function to each element in an RDD.

In [14]:
data_rdd.collect()

['0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True',
 '1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True',
 '3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False',
 '4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False',
 '5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False']

In [22]:
first_row = data_rdd.first()
first_row.split(',')[6] == "True"

True

In [23]:
def parse_row(row):
    fields = row.split(',')
    return (
     int(fields[0]),
        fields[1],
        fields[2],
        fields[3],
        fields[4],
        fields[5],
        fields[6]=='True'
    )

In [24]:
parsed_rdd = data_rdd.map(parse_row)

In [25]:
parsed_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

In [ ]:
## ADVANCE RDD OPERATIONS

In [ ]:
## Extract a field with Map() --Customer and city

In [27]:
name_city_rdd = parsed_rdd.map(lambda row: (row[1],row[2]))

In [28]:
name_city_rdd.collect()

[('Customer_0', 'Bangalore'),
 ('Customer_1', 'Hyderabad'),
 ('Customer_2', 'Ahmedabad'),
 ('Customer_3', 'Bangalore'),
 ('Customer_4', 'Bangalore'),
 ('Customer_5', 'Delhi')]

In [29]:
name_city_rdd.first()

('Customer_0', 'Bangalore')

In [ ]:
# Filter out Active Customers

In [30]:
active_customers = parsed_rdd.filter(lambda row: row[6] == True)

In [31]:
active_customers.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

In [ ]:
# distinct() - Transfomration

In [32]:
cities_rdd = parsed_rdd.map(lambda row:row[2]).distinct()

In [33]:
cities_rdd.collect()

['Hyderabad', 'Delhi', 'Bangalore', 'Ahmedabad']

In [34]:
# take()

cities_rdd.take(1)

['Hyderabad']

In [37]:
cities_rdd.take(3)

['Hyderabad', 'Delhi', 'Bangalore']

In [ ]:
# Reduce By Key - Transformation
# combines values for each key using an associative reduce function.

In [38]:
parsed_rdd.first()

(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True)

In [39]:
# we can chain the functions together
customers_per_city = parsed_rdd.map(lambda row : (row[2],1)).reduceByKey(lambda a, b: a+b)

In [40]:
customers_per_city

PythonRDD[31] at RDD at PythonRDD.scala:53

In [41]:
customers_per_city.collect()

[('Hyderabad', 1), ('Delhi', 1), ('Bangalore', 3), ('Ahmedabad', 1)]

In [ ]:
# CountByValue

In [42]:
cust_per_city = parsed_rdd.map(lambda row:row[2]).countByValue()

In [43]:
cust_per_city

defaultdict(int, {'Bangalore': 3, 'Hyderabad': 1, 'Ahmedabad': 1, 'Delhi': 1})

In [ ]:
# Important - ReducebyKey is a Transformation while CountByValue is an Action

In [ ]:
# Combine More Operations

In [45]:
parsed_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

In [50]:
# Cities with active customer

active_cities = parsed_rdd.filter(lambda row:row[6]).map(lambda row:row[2]).distinct()

active_cities.collect()

['Hyderabad', 'Bangalore', 'Ahmedabad']

In [51]:
# Count Active Customer By State

active_customer_State = parsed_rdd.filter(lambda row:row[6] == True).map(lambda row: (row[3], 1)).reduceByKey(lambda x,y : x + y)

active_customer_State.collect()


[('Delhi', 1), ('Karnataka', 1), ('West Bengal', 1)]

In [53]:
spark.stop()